# **Phase 1 : Data Loading**

In [1]:
import pandas as pd
import nltk
from nltk import word_tokenize, ISRIStemmer 
from nltk.corpus import stopwords
import string
import re
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
data= pd.read_csv('../data/ar_reviews_100k.tsv', sep="\t")
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
# data=data[:100000]


In [3]:
data['label'].value_counts()

label
Positive    33333
Negative    33333
Mixed       33333
Name: count, dtype: int64

In [4]:
data.isnull().sum()

label    0
text     0
dtype: int64

In [5]:
data.duplicated().sum()

np.int64(0)

# **Phase 2 : Data preprocessing**

In [7]:

def processing_text(text):
    Stopwords=set(stopwords.words("arabic"))
    text = re.sub(r'[^\u0621-\u064A\s]', ' ', text)
    tokens=word_tokenize(text)
    punctuations = set(string.punctuation)
    stemmer = ISRIStemmer()
    new_text=[ stemmer.stem(i) for i in tokens if (i not in Stopwords) and (i not in punctuations)]
    retunred_text=' '.join(new_text)
    return retunred_text

In [8]:
data['clean']=data['text'].apply(processing_text)

In [9]:
data.to_csv("../data/arabic_cleaned.csv")

## Most frequent words per class

In [10]:
from collections import Counter
count={}
for cls, group in data.groupby('label'):
    text = ' '.join(group['clean'])
    count[cls]=Counter(text.split())

for class_name, counts in count.items():
    print(f"\nClass {class_name}:")
    print(counts)


Class Mixed:
Counter({'كتب': 27124, 'روي': 13777, 'جدا': 10046, 'ان': 8801, 'قبل': 8781, 'غرف': 8638, 'وجد': 8617, 'ندق': 8509, 'وقع': 8163, 'حدث': 8121, 'عمل': 7711, 'انه': 7566, 'فى': 7546, 'جمل': 7538, 'فكر': 6756, 'علم': 6703, 'كثر': 6678, 'شخص': 6441, 'جيد': 6408, 'الل': 5567, 'جمع': 5422, 'نفس': 5404, 'شكل': 5289, 'كانت': 5097, 'خدم': 4962, 'عرف': 4932, 'نظف': 4898, 'شعر': 4708, 'قرب': 4703, 'اخر': 4668, 'عجب': 4496, 'نهي': 4483, 'قرء': 4385, 'نظر': 4333, 'قدم': 4207, 'دخل': 4002, 'فصل': 3988, 'قرأ': 3941, 'سلب': 3891, 'كلم': 3889, 'سلم': 3852, 'علي': 3775, 'وقف': 3735, 'طرق': 3715, 'رضي': 3478, 'الى': 3472, 'عبر': 3448, 'عدم': 3443, 'خلف': 3393, 'ذكر': 3376, 'كبر': 3370, 'نسب': 3210, 'وحد': 3197, 'بعد': 3163, 'حقق': 3153, 'او': 3111, 'قصة': 3054, 'رغم': 3049, 'فعل': 3032, 'كنت': 2936, 'سبب': 2868, 'حرم': 2845, 'مش': 2814, 'حمد': 2774, 'فضل': 2772, 'حية': 2767, 'عند': 2761, 'وضع': 2757, 'اول': 2747, 'فقط': 2733, 'نجم': 2724, 'خرج': 2717, 'رئع': 2708, 'طلب': 2673, 'وصل': 2601, 'س

# **Phase 3 : Convert to Numerical Data**

In [11]:
from sklearn.preprocessing import LabelEncoder
cv = TfidfVectorizer(max_features=10000)
x = cv.fit_transform(data['clean']).toarray()


y = data['label']
label = LabelEncoder()
y = label.fit_transform(y)

# **Phase 4 : train_test_split**

In [12]:

from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test = train_test_split(x,y,train_size=0.8)

# **Phase 5 : Model Selection**

In [14]:
from sklearn.naive_bayes import GaussianNB


model = GaussianNB()
model.fit(x_train, y_train)

GaussianNB()

# **Phase 6 : Model Evaluation**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(x_test)

print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

# **Phase 7 : Save Model**

In [16]:
import pickle

with open('../models/model_arabic.pkl', 'wb') as file:
    pickle.dump(model, file)

In [17]:
with open('../models/vectorizer_arabic.pkl', 'wb') as file:
    pickle.dump(cv, file)

In [18]:
with open('../models/LE_arabic.pkl', 'wb') as file:
    pickle.dump(label, file)